In [ ]:
import os
import shutil
import yaml
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO
from roboflow import Roboflow


rf = Roboflow(api_key="QWnrcDH9z1LGiIn7xLyX")
project = rf.workspace("mihailo").project("football-field-detection-f07vi-apxzb")
version = project.version(1)
dataset = version.download("yolov8")

yaml_path = os.path.join(dataset.location, "data.yaml")

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset.location, "train/images")
data['val'] = os.path.join(dataset.location, "valid/images")
data['test'] = os.path.join(dataset.location, "test/images")

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print(f"data.yaml fixed at: {yaml_path}")

print("Starting YOLOv8x-Pose Training")
!yolo task=pose \
      mode=train \
      model=yolov8x-pose.pt \
      data={yaml_path} \
      epochs=500 \
      patience=40 \
      batch=8 \
      imgsz=640 \
      mosaic=0.0 \
      plots=True \
      project=runs/pose \
      name=train_field_v8x


results_dir = 'runs/pose/train_field_v8x'
val_pred_path = f'{results_dir}/val_batch0_pred.jpg'

if os.path.exists(val_pred_path):
    plt.figure(figsize=(15, 15))
    plt.axis('off')
    plt.imshow(mpimg.imread(val_pred_path))
    plt.show()
else:
    print("No validation image found. Check if training completed.")

source_path = f'{results_dir}/weights/best.pt'
destination_dir = './exported_models' 
destination_file = os.path.join(destination_dir, 'keypoints_v8x_640.pt')

os.makedirs(destination_dir, exist_ok=True)

if os.path.exists(source_path):
    shutil.copy(source_path, destination_file)
    print(f"Success! Model saved to: {destination_file}")
else:
    print(f"Error: Could not find {source_path}")
